# **CS4120 Project: Analyzing Bias in Different Cuisine Type Restaurant Reviews (Yelp)**

### Melina Yang, Arpitha Coorg, Sheryl Cheng

## Yelp Restaurant & Cuisine Preprocessing

Filter businesses to restaurants, assign ethnic cuisine labels, merge reviews in chunks, balance samples, clean text, and export baseline data.

In [23]:
# imports
import gc
import os
import re
from pathlib import Path
import pandas as pd

In [24]:
# global variables
merged_data = "final_yelp_dataset.csv"

output_csv = "processed_reviews_baseline.csv"

max_per_cuisine = 10_000
sample_random_state = 42


### Filter for restaurants only

Keep businesses whose `categories` string contains **Restaurants** (case-insensitive).

In [25]:
# load dataset
merged_df = pd.read_csv(merged_data)

# business-level counts
n_business_total = merged_df["business_id"].nunique()

# keep rows where categories mention Restaurants
cat_mask = merged_df["categories"].fillna("").str.contains(
    "Restaurants", case=False, na=False, regex=False
)
reviews_rest = merged_df.loc[cat_mask].copy()
n_restaurants = reviews_rest["business_id"].nunique()

### Assign cuisine types

Match comma-separated `categories` against keywords; first cuisine in the fixed priority order wins.

In [26]:
# order matters: first matching cuisine in this list is assigned.
cuisine_mapping = [
    ("Chinese", ["Chinese", "Cantonese", "Szechuan", "Dim Sum"]),
    ("Japanese", ["Japanese", "Sushi", "Ramen", "Izakaya"]),
    ("Korean", ["Korean"]),
    ("Thai", ["Thai"]),
    ("Vietnamese", ["Vietnamese", "Pho"]),
    ("Indian", ["Indian", "Pakistani", "Bangladeshi", "Himalayan"]),
    ("Mexican", ["Mexican", "Tex-Mex"]),
    ("Italian", ["Italian"]),
    (
        "American",
        ["American (Traditional)", "American (New)", "Burgers"],
    ),
    ("French", ["French"]),
]


def assign_cuisine_type(categories) -> str:
    """
    If any keyword appears in categories string (case-insensitive),
    return first matching cuisine in cuisine_mapping order.
    Otherwise return 'Other'.
    """
    if pd.isna(categories) or not str(categories).strip():
        return "Other"
    text = str(categories).lower()
    for cuisine, keywords in cuisine_mapping:
        for kw in keywords:
            if kw.lower() in text:
                return cuisine
    return "Other"


# prefer precomputed cuisine if present; fallback to category-based mapping
if "cuisine" in reviews_rest.columns:
    reviews_rest["cuisine_type"] = reviews_rest["cuisine"].fillna("").astype(str).str.strip()
    reviews_rest.loc[reviews_rest["cuisine_type"] == "", "cuisine_type"] = reviews_rest.loc[
        reviews_rest["cuisine_type"] == "", "categories"
    ].map(assign_cuisine_type)
else:
    reviews_rest["cuisine_type"] = reviews_rest["categories"].map(assign_cuisine_type)

reviews_rest.loc[reviews_rest["cuisine_type"].isna(), "cuisine_type"] = "Other"


### Prepare review-level subset from merged dataset

Keep only rows where `cuisine_type != 'Other'`, keep each business’s Yelp **price tier** (`tier`: `$`, `$$`, …), and standardize column names.

In [ ]:
# build review-level table directly from merged dataset
reviews_merged = reviews_rest.copy()

# standardize column names to match downstream sentiment analysis pipeline
if "individual_stars" in reviews_merged.columns:
    reviews_merged = reviews_merged.rename(columns={"individual_stars": "stars"})

# keep only mapped cuisines (exclude "Other")
reviews_merged = reviews_merged.loc[reviews_merged["cuisine_type"] != "Other"].copy()

# ensure required columns exist
if "review_id" not in reviews_merged.columns:
    # deterministic synthetic ID if source does not provide review_id
    reviews_merged["review_id"] = [f"merged_review_{i}" for i in range(len(reviews_merged))]

# Yelp price tier ($ … $$$$) from Attributes; used later for exploratory breakdowns
if "tier" not in reviews_merged.columns:
    reviews_merged["tier"] = "Unknown"
else:
    reviews_merged["tier"] = reviews_merged["tier"].fillna("Unknown").astype(str).str.strip()
    reviews_merged.loc[reviews_merged["tier"] == "", "tier"] = "Unknown"

reviews_merged = reviews_merged.loc[reviews_merged["tier"] != "$$$$"].copy()

required_cols = ["review_id", "user_id", "business_id", "stars", "text", "date", "cuisine_type", "name", "city", "state", "tier"]
missing_cols = [c for c in required_cols if c not in reviews_merged.columns]

reviews_merged = reviews_merged[required_cols].copy()

del reviews_rest, merged_df
gc.collect()


211

### Balance: up to 10,000 reviews per cuisine

In [28]:
balanced_parts: list[pd.DataFrame] = []
for cuisine, g in reviews_merged.groupby("cuisine_type"):
    n = min(max_per_cuisine, len(g))
    balanced_parts.append(g.sample(n=n, random_state=sample_random_state))

balanced = pd.concat(balanced_parts, ignore_index=True)
del balanced_parts

del reviews_merged
gc.collect()


0

### Basic text cleaning and length filter

Lowercase, strip URLs, normalize whitespace; drop reviews with fewer than 10 words.

In [29]:
# match http(s) URLs and common www. spans
url_pattern = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
whitespace_pattern = re.compile(r"\s+")


def clean_review_text(text: str) -> str:
    if pd.isna(text):
        return ""
    t = str(text).lower()
    t = url_pattern.sub(" ", t)
    t = whitespace_pattern.sub(" ", t).strip()
    return t


balanced["stars"] = pd.to_numeric(balanced["stars"], errors="coerce")
balanced = balanced.dropna(subset=["stars"]).copy()

balanced["text_clean"] = balanced["text"].map(clean_review_text)
balanced["word_count"] = balanced["text_clean"].str.split().str.len().fillna(0).astype(int)
balanced = balanced.loc[balanced["word_count"] >= 10].copy()
balanced = balanced.drop(columns=["word_count"])


### Save processed reviews CSV

In [30]:
save_cols = [
    "review_id",
    "user_id",
    "business_id",
    "stars",
    "text",
    "text_clean",
    "date",
    "cuisine_type",
    "name",
    "city",
    "state",
    "tier",
]
balanced[save_cols].to_csv(output_csv, index=False)

---

## Baseline Sentiment Analysis

Load `processed_reviews_baseline.csv`, **drop `$$$$` tier**, run **VADER**, then report summaries **by cuisine** and **by cuisine & tier**, plus correlation tables, ANOVA, and American-vs-other pairwise tests on VADER.


In [31]:
# imports and downloads for nltk/vader
import os
import urllib.request
import zipfile
import ssl

# bypass SSL verification
ssl._create_default_https_context = ssl._create_unverified_context

# create directory
os.makedirs('nltk_data/sentiment', exist_ok=True)

# download nltk vader lexicon
url = 'https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/sentiment/vader_lexicon.zip'
zip_path = 'nltk_data/sentiment/vader_lexicon.zip'
urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('nltk_data/sentiment/')
    
import nltk
nltk.data.path.insert(0, './nltk_data')
from nltk.sentiment import SentimentIntensityAnalyzer

In [32]:
# other imports
import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
import plotly.express as px

### Load preprocessed data

In [33]:
df_sent = pd.read_csv("processed_reviews_baseline.csv")
if "tier" not in df_sent.columns:
    df_sent["tier"] = "Unknown"
# Same as transformer notebooks: exclude $$$$
df_sent = df_sent.loc[df_sent["tier"] != "$$$$"].copy()

### Apply VADER sentiment analysis

In [34]:
sia = SentimentIntensityAnalyzer()
vad = df_sent["text_clean"].fillna("").astype(str).apply(sia.polarity_scores)
v = pd.DataFrame(vad.tolist())
df_sent["vader_negative"] = v["neg"]
df_sent["vader_neutral"] = v["neu"]
df_sent["vader_positive"] = v["pos"]
df_sent["vader_compound"] = v["compound"]

### VADER summary tables (by cuisine, then by cuisine & tier)

In [35]:
df = df_sent.dropna(subset=["vader_compound", "stars"])


def summarize(g):
    vc = g["vader_compound"]
    st = g["stars"]
    n = len(vc)
    m = float(vc.mean())
    lo, hi = scipy_stats.t.interval(0.95, n - 1, loc=m, scale=scipy_stats.sem(vc)) if n > 1 else (m, m)
    return pd.Series(
        {
            "n_reviews": n,
            "mean_stars": round(float(st.mean()), 4),
            "mean_vader": round(m, 4),
            "std_vader": round(float(vc.std(ddof=1)), 4) if n > 1 else 0.0,
            "ci_lower_vader": round(float(lo), 4),
            "ci_upper_vader": round(float(hi), 4),
        }
    )


by_cuisine = df.groupby("cuisine_type").apply(summarize).reset_index().sort_values("mean_vader", ascending=False)
print("By cuisine")
display(by_cuisine)

by_cuisine_tier = df.groupby(["cuisine_type", "tier"]).apply(summarize).reset_index()
by_cuisine_tier["tier"] = pd.Categorical(by_cuisine_tier["tier"], ["$", "$$", "$$$", "Unknown"], ordered=True)
by_cuisine_tier = by_cuisine_tier.sort_values(["tier", "mean_vader"], ascending=[True, False])
print("By cuisine & tier")
display(by_cuisine_tier)

print("\nReview counts (cuisine × tier):")
print(df.groupby(["cuisine_type", "tier"]).size().unstack(fill_value=0))

By cuisine


,cuisine_type,n_reviews,mean_stars,mean_vader,std_vader,ci_lower_vader,ci_upper_vader
2,French,2317.0,4.1398,0.7234,0.4850,0.7036,0.7431
7,Korean,2385.0,4.1283,0.7175,0.4840,0.6980,0.7369
9,Thai,5367.0,4.1645,0.7150,0.4800,0.7022,0.7279
4,Indian,4893.0,4.1149,0.6882,0.5106,0.6739,0.7025
5,Italian,9993.0,3.9281,0.6665,0.5367,0.6560,0.6771
6,Japanese,9988.0,3.9263,0.6574,0.5400,0.6468,0.6680
3,Greek,3634.0,4.0272,0.6514,0.5428,0.6338,0.6691
8,Mexican,9993.0,3.8453,0.6374,0.5524,0.6265,0.6482
0,American,9991.0,3.7341,0.6095,0.5881,0.5980,0.6210
1,Chinese,9983.0,3.7625,0.5846,0.5851,0.5731,0.5961


By cuisine & tier


,cuisine_type,tier,n_reviews,mean_stars,mean_vader,std_vader,ci_lower_vader,ci_upper_vader
6,French,$,157.0,4.5987,0.7924,0.3705,0.7340,0.8509
21,Korean,$,225.0,4.3600,0.7479,0.3977,0.6956,0.8001
27,Thai,$,258.0,4.2752,0.7345,0.4184,0.6832,0.7858
12,Indian,$,299.0,4.0803,0.6605,0.5096,0.6025,0.7185
18,Japanese,$,240.0,4.0333,0.6569,0.5027,0.5929,0.7208
9,Greek,$,958.0,4.0532,0.6222,0.5635,0.5865,0.6579
24,Mexican,$,2830.0,3.8466,0.5852,0.5796,0.5639,0.6066
15,Italian,$,1086.0,3.7063,0.5540,0.6032,0.5180,0.5899
3,Chinese,$,2644.0,3.5495,0.4948,0.6300,0.4708,0.5189
0,American,$,1310.0,2.9420,0.3038,0.7291,0.2643,0.3433



Review counts (cuisine × tier):
tier             $    $$   $$$
cuisine_type                  
American      1310  7890   791
Chinese       2644  7222   117
French         157  1306   854
Greek          958  2578    98
Indian         299  4534    60
Italian       1086  7856  1051
Japanese       240  9053   695
Korean         225  2097    63
Mexican       2830  7113    50
Thai           258  5021    88


### Correlation: VADER compound vs. star rating

Overall Pearson correlation, then the same correlation **by cuisine** and **by cuisine & tier** (only groups with at least 3 usable rows).

In [36]:
ok = df_sent.dropna(subset=["vader_compound", "stars"])
r_all, p_all = scipy_stats.pearsonr(ok["vader_compound"], ok["stars"])
print(f"Overall: Pearson r = {r_all:.4f}, p = {p_all:.2e}\n")


def pearson_row(g):
    r, p = scipy_stats.pearsonr(g["vader_compound"], g["stars"])
    return pd.Series({"pearson_r": r, "p_value": p, "n": len(g)})


corr_c = (
    ok.groupby("cuisine_type")
    .filter(lambda x: len(x) >= 3)
    .groupby("cuisine_type")
    .apply(pearson_row)
    .reset_index()
)
print("By cuisine:")
display(corr_c.sort_values("cuisine_type"))

corr_ct = (
    ok.groupby(["cuisine_type", "tier"])
    .filter(lambda x: len(x) >= 3)
    .groupby(["cuisine_type", "tier"])
    .apply(pearson_row)
    .reset_index()
)
corr_ct["tier"] = pd.Categorical(corr_ct["tier"], ["$", "$$", "$$$", "Unknown"], ordered=True)
print("By cuisine & tier:")
display(corr_ct.sort_values(["tier", "cuisine_type"]))

Overall: Pearson r = 0.6968, p = 0.00e+00

By cuisine:


,cuisine_type,pearson_r,p_value,n
0,American,0.706955,0.000000e+00,9991.0
1,Chinese,0.702210,0.000000e+00,9983.0
2,French,0.660275,3.448218e-290,2317.0
3,Greek,0.715181,0.000000e+00,3634.0
4,Indian,0.706812,0.000000e+00,4893.0
5,Italian,0.682306,0.000000e+00,9993.0
6,Japanese,0.692681,0.000000e+00,9988.0
7,Korean,0.664913,2.530044e-304,2385.0
8,Mexican,0.692388,0.000000e+00,9993.0
9,Thai,0.690710,0.000000e+00,5367.0


By cuisine & tier:


,cuisine_type,tier,pearson_r,p_value,n
0,American,$,0.740698,3.437050e-228,1310.0
3,Chinese,$,0.723454,0.000000e+00,2644.0
6,French,$,0.607354,3.363300e-17,157.0
9,Greek,$,0.735618,6.767230e-164,958.0
12,Indian,$,0.668316,4.724401e-40,299.0
15,Italian,$,0.704956,6.357179e-164,1086.0
18,Japanese,$,0.716670,4.043379e-39,240.0
21,Korean,$,0.557518,8.923441e-20,225.0
24,Mexican,$,0.712647,0.000000e+00,2830.0
27,Thai,$,0.728119,6.975624e-44,258.0


### ANOVA: VADER compound across cuisines, then across tiers

One-way ANOVA on `vader_compound` (same outcome variable as in the summary tables).

In [37]:
gc = [g["vader_compound"].values for _, g in df_sent.groupby("cuisine_type")]
gt = [g["vader_compound"].values for _, g in df_sent.groupby("tier")]
f_c, p_c = scipy_stats.f_oneway(*gc)
f_t, p_t = scipy_stats.f_oneway(*gt)
print(f"ANOVA (VADER across cuisines): F = {f_c:.4f}, p = {p_c:.2e}")
print(f"ANOVA (VADER across tiers):    F = {f_t:.4f}, p = {p_t:.2e}")

ANOVA (VADER across cuisines): F = 43.6992, p = 6.35e-79
ANOVA (VADER across tiers):    F = 259.8737, p = 3.66e-113


### Pairwise tests: American vs. other cuisines (VADER)

Same style as the DistilBERT notebook: compare **American** to each other cuisine on mean `vader_compound` (Welch’s t-test).

In [38]:
am = df_sent.loc[df_sent["cuisine_type"] == "American", "vader_compound"].dropna()
print("American vs other cuisine (VADER compound)\n")
for cuisine, g in df_sent.groupby("cuisine_type"):
    if cuisine == "American":
        continue
    o = g["vader_compound"].dropna()
    t_stat, p_val = scipy_stats.ttest_ind(am, o, equal_var=False)
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
    print(
        f"American vs {cuisine:12}  diff(other−American)={o.mean() - am.mean():+.4f}  "
        f"t={t_stat:7.2f}  p={p_val:.4e}  {sig}"
    )

American vs other cuisine (VADER compound)

American vs Chinese       diff(other−American)=-0.0249  t=   3.00  p=2.7212e-03  **
American vs French        diff(other−American)=+0.1139  t=  -9.76  p=3.0025e-22  ***
American vs Greek         diff(other−American)=+0.0419  t=  -3.90  p=9.8186e-05  ***
American vs Indian        diff(other−American)=+0.0787  t=  -8.40  p=5.1939e-17  ***
American vs Italian       diff(other−American)=+0.0570  t=  -7.16  p=8.2228e-13  ***
American vs Japanese      diff(other−American)=+0.0479  t=  -6.00  p=1.9984e-09  ***
American vs Korean        diff(other−American)=+0.1080  t=  -9.37  p=1.1727e-20  ***
American vs Mexican       diff(other−American)=+0.0279  t=  -3.45  p=5.5649e-04  ***
American vs Thai          diff(other−American)=+0.1055  t= -11.98  p=6.4172e-33  ***


In [39]:
df_sent.to_csv("processed_reviews_with_sentiment.csv", index=False)

---

## Word Frequency and Stereotype Word Analysis

Analyze stereotype-word mentions and top word frequencies by cuisine.

In [40]:
import numpy as np
import pandas as pd
import re
from collections import Counter
from scipy.stats import chi2_contingency
import plotly.express as px

In [41]:
word_df = pd.read_csv("processed_reviews_with_sentiment.csv")
if "tier" not in word_df.columns:
    word_df["tier"] = "Unknown"
word_df = word_df.loc[word_df["tier"] != "$$$$"].copy()

### Define restaurant review stereotype word categories

In [42]:
# this was a gathered list from many sources off the internet and public opinion
stereotype_words = {
    "cheap": ["cheap", "inexpensive", "affordable", "budget"],
    "expensive": ["expensive", "pricey", "overpriced", "costly"],
    "authentic": ["authentic", "traditional", "genuine", "real", "old-school", "old school"],
    "dirty": ["dirty", "grimy", "filthy", "unsanitary", "gross"],
    "clean": ["clean", "spotless", "sanitary", "hygienic"],
    "exotic": ["exotic", "strange", "weird", "unusual"],
    "fresh": ["fresh", "quality"],
    "greasy": ["greasy", "oily"],
    "hidden_gem": ["hidden gem", "hole in the wall"],
    "must_try_hype": ["must-try", "must try", "to die for", "best ever", "best i've ever", "best i have ever"],
    "homestyle": ["homemade", "home-style", "homestyle"],
    "negative_texture": ["stale", "cold", "chewy", "rubber", "rubbery", "watery", "bland", "disappointing"],
    "luxury_sensual": ["decadent", "sensual", "sexy", "voluptuous", "voluptuously", "hedonistic", "divine", "transcendent"],
    "atmosphere_positive": ["cozy", "intimate", "buzzy", "romantic"],
    "atmosphere_negative": ["loud", "stuffy"],
    "kitschy_authenticity": ["plastic stools", "dirt floors", "kitschy"],
}

### Count stereotype word occurrences

In [43]:
text_series = word_df["text_clean"].fillna("").astype(str)
cuisines_sorted = sorted(word_df["cuisine_type"].dropna().unique().tolist())

records = []

for cuisine in cuisines_sorted:
    c_mask = word_df["cuisine_type"] == cuisine
    c_text = text_series.loc[c_mask]
    total = int(c_mask.sum())

    for category, words in stereotype_words.items():
        pattern = r"\b(?:" + "|".join(re.escape(w.lower()) for w in words) + r")\b"
        has_word = c_text.str.contains(pattern, regex=True, case=False, na=False)
        count = int(has_word.sum())
        frequency = count / total if total else 0.0

        records.append(
            {
                "cuisine": cuisine,
                "stereotype_category": category,
                "word": " | ".join(words),
                "count": count,
                "frequency": frequency,
            }
        )

stereo_freq_df = pd.DataFrame(records)

pivot_freq = stereo_freq_df.pivot(
    index="cuisine", columns="stereotype_category", values="frequency"
).fillna(0.0)

display(pivot_freq)

stereotype_category,atmosphere_negative,atmosphere_positive,authentic,cheap,clean,dirty,exotic,expensive,fresh,greasy,hidden_gem,homestyle,kitschy_authenticity,luxury_sensual,must_try_hype,negative_texture
cuisine,,,,,,,,,,,,,,,,
American,0.015014,0.012812,0.031228,0.013412,0.036433,0.019518,0.012011,0.036533,0.105595,0.011210,0.005905,0.011210,0.000200,0.006706,0.024822,0.088079
Chinese,0.004107,0.004007,0.085245,0.028048,0.046179,0.015727,0.014825,0.034659,0.176901,0.028448,0.010017,0.007212,0.000000,0.002504,0.019133,0.097967
French,0.014243,0.033664,0.053517,0.012085,0.018990,0.014674,0.011653,0.034959,0.125593,0.008632,0.011221,0.009495,0.000432,0.015537,0.033664,0.080708
Greek,0.006604,0.006329,0.084755,0.019263,0.044029,0.012108,0.011558,0.032196,0.192075,0.012383,0.006329,0.018987,0.000275,0.004953,0.027243,0.068520
Indian,0.002452,0.006744,0.088289,0.013284,0.044962,0.008992,0.010627,0.024320,0.132025,0.013489,0.009606,0.005314,0.000000,0.006131,0.027386,0.077662
Italian,0.012609,0.023717,0.065846,0.011208,0.024617,0.010507,0.011608,0.029921,0.152006,0.011608,0.010808,0.025718,0.000100,0.007205,0.029220,0.086761
Japanese,0.008010,0.010913,0.046256,0.019523,0.047857,0.016219,0.014618,0.037545,0.241690,0.008010,0.010713,0.002203,0.000000,0.004405,0.022827,0.073088
Korean,0.005870,0.007966,0.094340,0.020545,0.050314,0.009224,0.011740,0.043187,0.138784,0.010482,0.012579,0.006289,0.000000,0.002935,0.023899,0.072117
Mexican,0.011208,0.004803,0.098769,0.022015,0.035425,0.017612,0.014210,0.034424,0.151806,0.013309,0.012209,0.018513,0.000300,0.002402,0.022716,0.084559


### Chi-square tests, top words, comparison chart

In [44]:
# chi-square tests per stereotype category (present vs absent counts by cuisine)
n_per = word_df["cuisine_type"].value_counts()
cnt_pivot = stereo_freq_df.pivot(index="cuisine", columns="stereotype_category", values="count").fillna(0)
chi_rows = []
for cat in stereotype_words:
    col = cnt_pivot[cat]
    tbl = np.column_stack([col.values, n_per.reindex(col.index, fill_value=0).values - col.values])
    if tbl.sum() == 0 or tbl.shape[0] < 2 or tbl[:, 0].sum() == 0:
        chi2_stat, p_val = np.nan, np.nan
    else:
        chi2_stat, p_val, _, _ = chi2_contingency(tbl)
    chi_rows.append({"category": cat, "chi2_stat": chi2_stat, "p_value": p_val})

chi_df = pd.DataFrame(chi_rows)
display(chi_df)

# stop words
stop_words = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with",
    "is", "was", "were", "been", "be", "have", "has", "had", "do", "does", "did", "doing",
    "am", "are", "isn", "wasn", "weren", "don", "didn", "doesn", "won", "wouldn", "shouldn",
    "can", "could", "may", "might", "must", "shall", "will", "would", "should",
    "i", "me", "my", "mine", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours",
    "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself",
    "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "this", "that", "these", "those",
    "there", "here", "where", "when", "who", "whom", "whose", "which", "what", "why", "how",
    "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "only", "own", "same",
    "so", "than", "too", "very", "really", "just", "also", "still", "even", "much", "many", "lot", "lots",
    "not", "no", "nor", "if", "then", "because", "while", "into", "onto", "over", "under", "again",
    "up", "down", "out", "off", "as", "from", "by", "about", "after", "before", "during", "through",
    "food", "restaurant", "place", "good", "great", "nice", "bad", "best", "better", "worst",
    "service", "staff", "menu", "ordered", "order", "got", "get", "came", "come", "went", "go",
    "one", "two", "three", "first", "last", "always", "never", "ever", "make", "made",
}

# extra non-informative contractions/fragments common in Yelp text tokenization
stop_words.update({"im", "ive", "id", "ill", "youre", "youve", "youll", "theyre", "theyve", "weve", "cant", "couldnt", "didnt", "doesnt", "dont", "isnt", "wasnt", "werent", "wouldnt", "shouldnt", "wont", "thats", "theres", "heres"})

# find top words for each cuisine
word_pattern = re.compile(r"[a-z]+")
top_words_map = {}
for cuisine in cuisines_sorted:
    c_text = text_series[word_df["cuisine_type"] == cuisine]
    counter = Counter()
    for t in c_text:
        tokens = word_pattern.findall(t.lower())
        tokens = [w for w in tokens if len(w) >= 3 and w not in stop_words]
        counter.update(tokens)
    top20 = counter.most_common(20)
    top_words_map[cuisine] = top20

top_words_rows = []
for cuisine, pairs in top_words_map.items():
    for rank, (word, count) in enumerate(pairs, start=1):
        top_words_rows.append({"cuisine": cuisine, "rank": rank, "word": word, "count": int(count)})

top_words_df = pd.DataFrame(top_words_rows)
display(top_words_df)

# compare word frequencies for specific words
compare_words = ["cheap", "authentic", "dirty", "exotic", "greasy"]
compare_rows = []
for cuisine in cuisines_sorted:
    c_mask = word_df["cuisine_type"] == cuisine
    total = int(c_mask.sum())
    c_text = text_series.loc[c_mask]
    for w in compare_words:
        has = c_text.str.contains(rf"\b{re.escape(w)}\b", regex=True, case=False, na=False)
        freq = 0.0 if total == 0 else float(has.mean())
        compare_rows.append({"cuisine": cuisine, "word": w, "frequency": freq})

compare_df = pd.DataFrame(compare_rows)
plot_df = compare_df.pivot(index="cuisine", columns="word", values="frequency").fillna(0.0)

plot_pct = plot_df * 100
long_compare = plot_pct.reset_index().melt(id_vars="cuisine", var_name="word", value_name="pct")
fig_words = px.bar(
    long_compare,
    x="cuisine",
    y="pct",
    color="word",
    barmode="group",
    title="Stereotype word frequency by cuisine",
    labels={"cuisine": "Cuisine", "pct": "Frequency (% of reviews)", "word": "Word"},
)
fig_words.update_layout(xaxis_tickangle=-45)
fig_words.show()


,category,chi2_stat,p_value
0,cheap,118.376326,2.871342e-21
1,expensive,34.018740,8.864918e-05
2,authentic,584.955869,3.542738e-120
3,dirty,68.863624,2.542381e-11
4,clean,179.868158,5.398300e-34
5,exotic,13.233284,1.523314e-01
6,fresh,821.310139,5.495452e-171
7,greasy,196.519729,1.776403e-37
8,hidden_gem,33.144794,1.260171e-04
9,must_try_hype,36.255103,3.571959e-05


,cuisine,rank,word,count
0,American,1,time,2923
1,American,2,back,2894
2,American,3,like,2556
3,American,4,delicious,1948
4,American,5,chicken,1880
...,...,...,...,...
195,Thai,16,definitely,893
196,Thai,17,amazing,892
197,Thai,18,dish,850
198,Thai,19,fresh,822
